In [ ]:

# Advanced Customer Segmentation Project


import os
os.environ["OMP_NUM_THREADS"] = "1"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error


# Load Dataset


data = pd.read_csv("Mall_Customers.csv")

print("First 5 Rows")
print(data.head())

print("\nDataset Info")
print(data.info())


# Data Preprocessing


le = LabelEncoder()
data["Genre"] = le.fit_transform(data["Genre"])


# Exploratory Data Analysis


plt.figure()
sns.histplot(data["Age"], bins=20)
plt.title("Age Distribution")
plt.show()

plt.figure()
sns.scatterplot(x="Annual Income (k$)", y="Spending Score (1-100)", data=data)
plt.title("Income vs Spending Score")
plt.show()

plt.figure()
sns.boxplot(x="Genre", y="Spending Score (1-100)", data=data)
plt.title("Gender vs Spending Score")
plt.show()

plt.figure()
sns.heatmap(data.corr(), annot=True)
plt.title("Correlation Matrix")
plt.show()


# Customer Segmentation (KMeans)


X_cluster = data[["Annual Income (k$)", "Spending Score (1-100)"]]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Elbow Method
wcss = []

for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.figure()
plt.plot(range(1, 11), wcss)
plt.title("Elbow Method")
plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")
plt.show()

# Final Model
kmeans = KMeans(n_clusters=5, random_state=42)
data["Cluster"] = kmeans.fit_predict(X_scaled)

# Cluster Visualization
plt.figure()
sns.scatterplot(
    x=data["Annual Income (k$)"],
    y=data["Spending Score (1-100)"],
    hue=data["Cluster"],
    palette="Set1"
)
plt.title("Customer Segmentation")
plt.show()


# Cluster Profiling


cluster_profile = data.groupby("Cluster").mean(numeric_only=True)

print("\nCluster Profile")
print(cluster_profile)


# Spending Prediction (Random Forest)


X = data[["Age", "Genre", "Annual Income (k$)"]]
y = data["Spending Score (1-100)"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor()

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("\nModel Performance")
print("R2 Score:", r2_score(y_test, predictions))
print("RMSE:", np.sqrt(mean_squared_error(y_test, predictions)))

FileNotFoundError: [Errno 2] No such file or directory: 'Mall_Customers.csv'